# DPT Tactile Depth/Normal Training (DINOv2 Encoder)

Train a DPT decoder on frozen DINOv2 encoder for tactile depth + surface normal prediction.

## Dataset Preparation

1. Upload `renders.tar` (~25GB) to Google Drive
2. Set `DRIVE_TAR` path in the next cell

Dataset structure (nested layout):
```
renders/
  <object>/
    session_XXX/
      sensor_XXXX/
        calibration/   0000.png (ref) + 0001-0018.png
        samples/       0000.png, 0001.png, ...
        dmaps/         0000_gt.png, ...  (depth GT)
        norms/         0000_gt.png, ...  (normal GT)
```

**Runtime**: GPU T4 (free) or A100/V100 (Pro). AMP enabled, bs=64 fits T4.

In [ ]:
# ── 1. Environment Setup ─────────────────────────────────────────
import subprocess, sys

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install deps
!pip install -q albumentations opencv-python-headless

In [ ]:
# ── 2. Extract Dataset ────────────────────────────────────────────
import os

# ★ Change this to your .tar path on Google Drive
DRIVE_TAR = '/content/drive/MyDrive/renders.tar'
DATA_ROOT = '/content/renders'    # extracted destination (local SSD)

if not os.path.isdir(DATA_ROOT):
    print(f'Extracting {DRIVE_TAR} ...')
    !tar xf "{DRIVE_TAR}" -C /content/
    print('Done.')
else:
    print(f'{DATA_ROOT} already exists, skipping extraction.')

# Verify
objects = sorted([d for d in os.listdir(DATA_ROOT)
                  if os.path.isdir(os.path.join(DATA_ROOT, d))])
print(f'Objects ({len(objects)}): {objects}')

In [ ]:
# ── 3. Configuration ─────────────────────────────────────────────
from types import SimpleNamespace

CFG = SimpleNamespace(
    # ─ data ─
    data_path       = DATA_ROOT,
    val_objects      = ['edge', 'hex_key', 'pattern_31_rod'],
    use_gt_norm      = True,

    # ─ encoder ─
    encoder          = 'dinov2',          # 'dinov2' or 'sitr'
    dinov2_model     = 'dinov2_vitb14',
    sitr_weights     = None,              # path to SITR .pth (only if encoder='sitr')
    calibration_config = 0,               # 0 for DINOv2, 19 for raw SITR

    # ─ decoder ─
    dpt_features     = 256,
    dropout          = 0.1,

    # ─ augmentation ─
    tactile_augment  = True,

    # ─ loss ─
    lambda_depth     = 1.0,
    lambda_normal    = 1.0,
    lambda_grad      = 0.0,

    # ─ training ─
    epochs           = 200,
    batch_size       = 64,
    lr               = 1e-4,
    min_lr           = 1e-6,
    warmup_epochs    = 5,
    weight_decay     = 0.1,
    scheduler        = 'plateau',         # 'plateau' or 'cosine'
    plateau_patience = 10,
    plateau_factor   = 0.5,
    early_stop       = 30,
    amp              = True,
    num_workers      = 0,
    seed             = 42,

    # ─ save ─
    save_path        = '/content/drive/MyDrive/dpt_checkpoints/dpt_dinov2_colab',
    save_every       = 10,
    resume           = None,              # path to latest.pth for resume
)

# Auto-config
if CFG.encoder == 'dinov2':
    CFG.calibration_config = 0
    CFG.raw_input = True
elif CFG.encoder == 'sitr':
    CFG.raw_input = True
    if CFG.calibration_config == 18:
        CFG.calibration_config = 19

os.makedirs(CFG.save_path, exist_ok=True)
print('Config ready. Save path:', CFG.save_path)

In [ ]:
# ── 4. Model Definitions ─────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F


class Reassemble(nn.Module):
    def __init__(self, embed_dim, out_channels, scale_factor):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.proj = nn.Conv2d(embed_dim, out_channels, kernel_size=1)
        self.scale_factor = scale_factor

    def forward(self, tokens):
        B, N, D = tokens.shape
        h = w = int(N ** 0.5)
        tokens = self.norm(tokens)
        x = tokens.permute(0, 2, 1).reshape(B, D, h, w)
        x = self.proj(x)
        if self.scale_factor != 1.0:
            x = F.interpolate(x, scale_factor=self.scale_factor,
                              mode='bilinear', align_corners=True)
        return x


class ResidualConvUnit(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.conv1 = nn.Conv2d(features, features, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(features)
        self.conv2 = nn.Conv2d(features, features, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(features)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.relu(x)
        out = self.conv1(out)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        return out + x


class FeatureFusionBlock(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.rcu1 = ResidualConvUnit(features)
        self.rcu2 = ResidualConvUnit(features)

    def forward(self, x, skip=None):
        if skip is not None:
            x = x + self.rcu1(skip)
        x = self.rcu2(x)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=True)
        return x


class DPTDecoder(nn.Module):
    def __init__(self, embed_dim=768, features=256, dropout=0.0):
        super().__init__()
        self.reassemble = nn.ModuleList([
            Reassemble(embed_dim, features, scale_factor=4.0),
            Reassemble(embed_dim, features, scale_factor=2.0),
            Reassemble(embed_dim, features, scale_factor=1.0),
            Reassemble(embed_dim, features, scale_factor=0.5),
        ])
        self.fusion = nn.ModuleList([FeatureFusionBlock(features) for _ in range(4)])
        self.drop = nn.Dropout2d(p=dropout)
        self.depth_head = nn.Sequential(
            nn.Conv2d(features, features // 2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(features // 2, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1),
        )
        self.normal_head = nn.Sequential(
            nn.Conv2d(features, features // 2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(features // 2, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, 1),
        )

    def forward(self, features):
        maps = [r(f) for f, r in zip(features, self.reassemble)]
        x = self.fusion[0](maps[3])
        x = self.fusion[1](x, maps[2])
        x = self.fusion[2](x, maps[1])
        x = self.fusion[3](x, maps[0])
        x = self.drop(x)
        depth = self.depth_head(x)
        normal = self.normal_head(x)
        return depth, normal


class DINOv2MultiScale(nn.Module):
    def __init__(self, model_name='dinov2_vitb14', layer_indices=(2, 5, 8, 11)):
        super().__init__()
        self.dinov2 = torch.hub.load('facebookresearch/dinov2', model_name)
        self.layer_indices = sorted(layer_indices)
        self.embed_dim = self.dinov2.embed_dim
        for p in self.dinov2.parameters():
            p.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        self.dinov2.eval()
        return self

    def forward(self, x):
        features = self.dinov2.get_intermediate_layers(
            x, n=self.layer_indices, reshape=False, return_class_token=False)
        return list(features)


class DINOv2WithDPT(nn.Module):
    def __init__(self, model_name='dinov2_vitb14', features=256,
                 layer_indices=(2, 5, 8, 11), dropout=0.0):
        super().__init__()
        self.encoder = DINOv2MultiScale(model_name, layer_indices)
        embed_dim = self.encoder.embed_dim
        self.decoder = DPTDecoder(embed_dim, features, dropout=dropout)

    def forward(self, x, c=None):
        features = self.encoder(x)
        depth, normal = self.decoder(features)
        h, w = x.shape[2:]
        if depth.shape[2:] != (h, w):
            depth = F.interpolate(depth, size=(h, w), mode='bilinear', align_corners=True)
            normal = F.interpolate(normal, size=(h, w), mode='bilinear', align_corners=True)
        return {'depth': depth, 'normal': normal}


print('Model definitions loaded.')

In [ ]:
# ── 5. Dataset & Augmentation ────────────────────────────────────
import numpy as np
import cv2
import glob as _glob
import os.path as osp
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as T

# Normalization constants
sample_mu = [-1.2223, -1.8114, -1.7090]
sample_std = [11.7932, 12.7956, 13.6452]
dmap_mu = [8.8114]
dmap_std = [30.3624]
norm_mu = [126.9855, 127.2061, 247.7740]
norm_std = [25.1953, 25.5532, 22.1101]
raw_mu = [127.5, 127.5, 127.5]
raw_std = [127.5, 127.5, 127.5]
imagenet_mu = [123.675, 116.28, 103.53]
imagenet_std = [58.395, 57.12, 57.375]

DEFAULT_AUGMENT_PARAMS = {
    'gain': 0.5, 'bias': 45.0, 'grad': 0.7, 'bright': 25.0,
    'resid': 20.0, 'noise': 6.0, 'rot_deg': 15.0,
    'hflip': True, 'vflip': True,
}


class TactileAugment:
    def __init__(self, params=None):
        self.p = {**DEFAULT_AUGMENT_PARAMS, **(params or {})}

    def __call__(self, sample_diff, calib_diffs, depth, normal):
        p = self.p
        H, W = sample_diff.shape[:2]

        # Photometric (sample only)
        if p['gain'] > 0:
            g = np.random.uniform(1 - p['gain'], 1 + p['gain'], size=(1, 1, 3)).astype(np.float32)
            sample_diff = sample_diff * g
        if p['bias'] > 0:
            b = np.random.uniform(-p['bias'], p['bias'], size=(1, 1, 3)).astype(np.float32)
            sample_diff = sample_diff + b
        if p['bright'] > 0:
            sample_diff = sample_diff + np.float32(np.random.uniform(-p['bright'], p['bright']))
        if p['grad'] > 0:
            angle = np.random.uniform(0, 2 * np.pi)
            ys = np.linspace(-1, 1, H, dtype=np.float32).reshape(-1, 1)
            xs = np.linspace(-1, 1, W, dtype=np.float32).reshape(1, -1)
            grad_map = np.float32(np.cos(angle)) * xs + np.float32(np.sin(angle)) * ys
            amp = np.random.uniform(0, p['grad'], size=(1, 1, 3)).astype(np.float32)
            sample_diff = sample_diff + grad_map[..., None] * amp * np.float32(50.0)
        if p['resid'] > 0:
            raw = np.random.randn(16, 16, 3).astype(np.float32)
            smooth = cv2.resize(raw, (W, H), interpolation=cv2.INTER_LINEAR)
            smooth = cv2.GaussianBlur(smooth, (0, 0), sigmaX=H / 8.0)
            std = np.float32(smooth.std())
            if std > 1e-6:
                smooth = smooth / std * np.float32(p['resid'])
            sample_diff = sample_diff + smooth
        if p['noise'] > 0:
            noise = np.random.normal(0, p['noise'], sample_diff.shape).astype(np.float32)
            sample_diff = sample_diff + noise

        # Geometric (all images)
        do_hflip = p['hflip'] and np.random.random() < 0.5
        do_vflip = p['vflip'] and np.random.random() < 0.5
        rot_angle = 0.0
        if p['rot_deg'] > 0:
            rot_angle = np.random.uniform(-p['rot_deg'], p['rot_deg'])

        if do_hflip:
            sample_diff = np.ascontiguousarray(sample_diff[:, ::-1])
            calib_diffs = [np.ascontiguousarray(c[:, ::-1]) for c in calib_diffs]
            if depth is not None:
                depth = np.ascontiguousarray(depth[:, ::-1])
            if normal is not None:
                normal = np.ascontiguousarray(normal[:, ::-1])
                normal[:, :, 0] = 255.0 - normal[:, :, 0]

        if do_vflip:
            sample_diff = np.ascontiguousarray(sample_diff[::-1])
            calib_diffs = [np.ascontiguousarray(c[::-1]) for c in calib_diffs]
            if depth is not None:
                depth = np.ascontiguousarray(depth[::-1])
            if normal is not None:
                normal = np.ascontiguousarray(normal[::-1])
                normal[:, :, 1] = 255.0 - normal[:, :, 1]

        if abs(rot_angle) > 0.5:
            M = cv2.getRotationMatrix2D((W / 2, H / 2), rot_angle, 1.0)
            flags = cv2.INTER_LINEAR
            border = cv2.BORDER_REFLECT_101
            sample_diff = cv2.warpAffine(sample_diff, M, (W, H), flags=flags, borderMode=border)
            calib_diffs = [cv2.warpAffine(c, M, (W, H), flags=flags, borderMode=border) for c in calib_diffs]
            if depth is not None:
                depth = cv2.warpAffine(depth, M, (W, H), flags=flags, borderMode=border)
            if normal is not None:
                normal = cv2.warpAffine(normal, M, (W, H), flags=flags, borderMode=border)
                rad = np.radians(rot_angle)
                cos_a, sin_a = np.float32(np.cos(rad)), np.float32(np.sin(rad))
                nx = normal[:, :, 0] / 127.5 - 1.0
                ny = normal[:, :, 1] / 127.5 - 1.0
                normal[:, :, 0] = np.clip((cos_a * nx + sin_a * ny + 1.0) * 127.5, 0, 255)
                normal[:, :, 1] = np.clip((-sin_a * nx + cos_a * ny + 1.0) * 127.5, 0, 255)

        return sample_diff, calib_diffs, depth, normal


class TactileDataset(Dataset):
    """Nested tactile dataset for gs_blender renders."""

    def __init__(self, path, augment=False, transforms=None,
                 dmap_transforms=None, norm_transforms=None,
                 calibration_config=0, num_samples=None,
                 use_gt_norm=False, include_objects=None,
                 raw_input=True, tactile_augment=False):
        self.path = path
        self.transforms = transforms
        self.dmap_transforms = dmap_transforms
        self.norm_transforms = norm_transforms
        self.norm_suffix = '_gt' if use_gt_norm else ''
        self._calib_cache = {}
        self.raw_input = raw_input

        if calibration_config == 0:  self.calib_list = []
        elif calibration_config == 19: self.calib_list = list(range(0, 19))
        elif calibration_config == 18: self.calib_list = list(range(1, 19))
        else: self.calib_list = []

        units = sorted(_glob.glob(osp.join(path, '*', 'session_*', 'sensor_*')))
        if not units:
            units = sorted(_glob.glob(osp.join(path, 'sensor_*')))
        units = [u for u in units
                 if osp.isdir(osp.join(u, 'samples')) and osp.isdir(osp.join(u, 'calibration'))]

        if include_objects is not None:
            incl = set(include_objects)
            units = [u for u in units
                     if osp.basename(osp.dirname(osp.dirname(u))) in incl]

        if not units:
            raise RuntimeError(f'No units found in {path} (include_objects={include_objects})')

        self.units = units
        self.objects = sorted({osp.basename(osp.dirname(osp.dirname(u))) for u in units})

        samp = [f for f in os.listdir(osp.join(units[0], 'samples'))
                if f.endswith('.png') and '_gt' not in f]
        avail = len(samp)
        self.samples_per_unit = min(num_samples, avail) if num_samples else avail
        self.tactile_aug = TactileAugment() if tactile_augment else None

    def __len__(self):
        return len(self.units) * self.samples_per_unit

    def _get_calib(self, unit):
        cached = self._calib_cache.get(unit)
        if cached is None:
            cal_dir = osp.join(unit, 'calibration')
            ref = np.array(Image.open(osp.join(cal_dir, '0000.png')))
            calib = [np.array(Image.open(osp.join(cal_dir, f'{i:04d}.png')))
                     for i in range(1, 19)]
            cached = (ref, calib)
            self._calib_cache[unit] = cached
        return cached

    def __getitem__(self, index):
        unit_idx = index // self.samples_per_unit
        sample_idx = index % self.samples_per_unit
        unit = self.units[unit_idx]
        ref_img, calib_raw = self._get_calib(unit)

        sample = np.array(Image.open(osp.join(unit, 'samples', f'{sample_idx:04d}.png')))

        if self.raw_input:
            sample_f = sample.astype(np.float32)
            all_imgs = [ref_img.astype(np.float32)] + [c.astype(np.float32) for c in calib_raw]
            calib_imgs = [all_imgs[i] for i in self.calib_list]
        else:
            ref_f = ref_img.astype(np.float32)
            sample_f = sample.astype(np.float32) - ref_f
            calib_imgs = [(calib_raw[i - 1].astype(np.float32) - ref_f)
                          for i in self.calib_list]

        depth = normal = None
        try:
            dmap_path = osp.join(unit, 'dmaps', f'{sample_idx:04d}{self.norm_suffix}.png')
            depth = np.array(Image.open(dmap_path), dtype=np.float32)
            norm_path = osp.join(unit, 'norms', f'{sample_idx:04d}{self.norm_suffix}.png')
            normal = np.array(Image.open(norm_path), dtype=np.float32)
        except Exception:
            pass

        if self.tactile_aug is not None:
            sample_f, calib_imgs, depth, normal = self.tactile_aug(
                sample_f, calib_imgs, depth, normal)

        sample_t = self.transforms(sample_f)
        calib_t = torch.cat([self.transforms(c) for c in calib_imgs]) if calib_imgs else torch.empty(0)
        dmap_t = self.dmap_transforms(depth) if depth is not None else None
        norm_t = self.norm_transforms(normal) if normal is not None else None

        return {'sample': sample_t, 'calibration': calib_t,
                'dmap': dmap_t, 'norm': norm_t, 'idx': torch.tensor(index)}


print('Dataset classes loaded.')

In [ ]:
# ── 6. Training Utilities ────────────────────────────────────────
import time
import math
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm as tqdm_nb

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = self.avg = self.sum = self.count = 0.0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


class WarmupThenPlateau:
    def __init__(self, optimizer, warmup_epochs, warmup_lr, base_lr,
                 factor=0.5, plateau_patience=10, min_lr=1e-7):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.warmup_lr = warmup_lr
        self.base_lr = base_lr
        self.group_base_lrs = [pg['lr'] for pg in optimizer.param_groups]
        self.plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=factor,
            patience=plateau_patience, min_lr=min_lr)
        self._epoch = 0
        for pg in self.optimizer.param_groups:
            pg['lr'] = warmup_lr * (pg['lr'] / base_lr) if base_lr > 0 else warmup_lr

    def get_last_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]

    def step(self, val_loss=None):
        self._epoch += 1
        if self._epoch <= self.warmup_epochs:
            frac = self._epoch / max(1, self.warmup_epochs)
            for pg, target_lr in zip(self.optimizer.param_groups, self.group_base_lrs):
                start_lr = self.warmup_lr * (target_lr / self.base_lr) if self.base_lr > 0 else self.warmup_lr
                pg['lr'] = start_lr + (target_lr - start_lr) * frac
        else:
            self.plateau.step(val_loss)

    def state_dict(self):
        return {'epoch': self._epoch, 'plateau': self.plateau.state_dict()}
    def load_state_dict(self, d):
        self._epoch = d['epoch']
        self.plateau.load_state_dict(d['plateau'])


def gradient_loss(pred, gt):
    dx_pred = pred[:, :, :, 1:] - pred[:, :, :, :-1]
    dy_pred = pred[:, :, 1:, :] - pred[:, :, :-1, :]
    dx_gt = gt[:, :, :, 1:] - gt[:, :, :, :-1]
    dy_gt = gt[:, :, 1:, :] - gt[:, :, :-1, :]
    return F.l1_loss(dx_pred, dx_gt) + F.l1_loss(dy_pred, dy_gt)


def train_one_epoch(model, loader, optimizer, scaler,
                    depth_crit, normal_crit, cfg, device, epoch):
    model.train()
    loss_m, depth_m, norm_m = AverageMeter(), AverageMeter(), AverageMeter()
    t0 = time.time()

    for step, batch in enumerate(loader):
        imgs = batch['sample'].to(device, non_blocking=True)
        calibs = batch['calibration'].to(device, non_blocking=True)
        gt_depth = batch['dmap'].to(device, non_blocking=True)
        gt_norm = batch['norm'].to(device, non_blocking=True)
        B = imgs.size(0)

        optimizer.zero_grad()
        with autocast('cuda', enabled=cfg.amp):
            out = model(imgs, calibs)
            l_depth = depth_crit(out['depth'], gt_depth)
            l_normal = normal_crit(out['normal'], gt_norm)
            loss = cfg.lambda_depth * l_depth + cfg.lambda_normal * l_normal
            if cfg.lambda_grad > 0:
                loss = loss + cfg.lambda_grad * gradient_loss(out['depth'], gt_depth)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        loss_m.update(loss.item(), B)
        depth_m.update(l_depth.item(), B)
        norm_m.update(l_normal.item(), B)

        if step % 50 == 0:
            print(f'  [epoch {epoch:03d} | step {step:04d}/{len(loader):04d}]'
                  f'  loss={loss_m.avg:.4f}  depth={depth_m.avg:.4f}'
                  f'  normal={norm_m.avg:.4f}  ({time.time()-t0:.1f}s)')

    return loss_m.avg, depth_m.avg, norm_m.avg


@torch.no_grad()
def validate(model, loader, depth_crit, normal_crit, cfg, device):
    model.eval()
    loss_m, depth_m, norm_m = AverageMeter(), AverageMeter(), AverageMeter()

    for batch in loader:
        imgs = batch['sample'].to(device, non_blocking=True)
        calibs = batch['calibration'].to(device, non_blocking=True)
        gt_depth = batch['dmap'].to(device, non_blocking=True)
        gt_norm = batch['norm'].to(device, non_blocking=True)

        with autocast('cuda', enabled=cfg.amp):
            out = model(imgs, calibs)
            l_depth = depth_crit(out['depth'], gt_depth)
            l_normal = normal_crit(out['normal'], gt_norm)
            loss = cfg.lambda_depth * l_depth + cfg.lambda_normal * l_normal

        loss_m.update(loss.item(), imgs.size(0))
        depth_m.update(l_depth.item(), imgs.size(0))
        norm_m.update(l_normal.item(), imgs.size(0))

    return loss_m.avg, depth_m.avg, norm_m.avg


def plot_loss_curves(history, save_path):
    epochs = history['epochs']
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle('DPT Training Curves', fontsize=14, fontweight='bold')

    ax = axes[0, 0]
    ax.plot(epochs, history['train_loss'], color='#2563eb', lw=1.8, label='Train')
    ax.plot(epochs, history['val_loss'], color='#dc2626', lw=1.8, ls='--', label='Val')
    best_idx = history['val_loss'].index(min(history['val_loss']))
    ax.scatter(epochs[best_idx], history['val_loss'][best_idx],
               color='#dc2626', s=80, zorder=5,
               label=f'Best {history["val_loss"][best_idx]:.4f}')
    ax.set_title('Total Loss'); ax.set_xlabel('Epoch'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(epochs, history['l_depth'], color='#16a34a', lw=1.8)
    ax.set_title('Depth Loss'); ax.set_xlabel('Epoch'); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(epochs, history['l_normal'], color='#9333ea', lw=1.8)
    ax.set_title('Normal Loss'); ax.set_xlabel('Epoch'); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    ax.plot(epochs, history['lr'], color='#ea580c', lw=1.8)
    ax.set_title('Learning Rate'); ax.set_xlabel('Epoch'); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig.savefig(osp.join(save_path, 'loss_curve.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)


print('Training utilities loaded.')

In [ ]:
# ── 7. Build Dataset & Model ─────────────────────────────────────
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Transforms
if CFG.encoder == 'dinov2':
    img_xform = T.Compose([T.ToTensor(), T.Normalize(mean=imagenet_mu, std=imagenet_std)])
else:
    img_xform = T.Compose([T.ToTensor(), T.Normalize(mean=raw_mu, std=raw_std)])
dmap_xform = T.Compose([T.ToTensor(), T.Normalize(mean=dmap_mu, std=dmap_std)])
norm_xform = T.Compose([T.ToTensor(), T.Normalize(mean=norm_mu, std=norm_std)])

# Split by object
all_objs = sorted({osp.basename(osp.dirname(osp.dirname(p)))
                    for p in _glob.glob(osp.join(CFG.data_path, '*', 'session_*', 'sensor_*'))})
val_objs = CFG.val_objects
train_objs = [o for o in all_objs if o not in set(val_objs)]
print(f'Train objects ({len(train_objs)}): {train_objs}')
print(f'Val objects ({len(val_objs)}): {val_objs}')

def build_ds(augment, objects):
    return TactileDataset(
        path=CFG.data_path, augment=augment,
        transforms=img_xform, dmap_transforms=dmap_xform, norm_transforms=norm_xform,
        calibration_config=CFG.calibration_config,
        use_gt_norm=CFG.use_gt_norm, include_objects=objects,
        raw_input=CFG.raw_input,
        tactile_augment=augment and CFG.tactile_augment)

train_ds = build_ds(True, train_objs)
val_ds = build_ds(False, val_objs)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=CFG.num_workers, pin_memory=True,
                          drop_last=True, persistent_workers=(CFG.num_workers > 0),
                          prefetch_factor=4 if CFG.num_workers > 0 else None)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                        num_workers=CFG.num_workers, pin_memory=True,
                        persistent_workers=(CFG.num_workers > 0),
                        prefetch_factor=4 if CFG.num_workers > 0 else None)

print(f'Train: {len(train_ds):,} samples ({len(train_loader)} batches)')
print(f'Val:   {len(val_ds):,} samples ({len(val_loader)} batches)')

# Sanity check
s0 = train_ds[0]
assert s0['dmap'] is not None, 'GT depth not found'
assert s0['norm'] is not None, 'GT normal not found'
print(f'GT shapes: dmap={s0["dmap"].shape}, norm={s0["norm"].shape}')

# Model
print(f'\nBuilding {CFG.encoder} encoder...')
model = DINOv2WithDPT(
    model_name=CFG.dinov2_model, features=CFG.dpt_features, dropout=CFG.dropout
).to(device)

enc_p = sum(p.numel() for p in model.encoder.parameters())
dec_p = sum(p.numel() for p in model.decoder.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Encoder: {enc_p/1e6:.1f}M (frozen)')
print(f'Decoder: {dec_p/1e6:.1f}M (trainable)')
print(f'Total trainable: {trainable/1e6:.1f}M')

In [ ]:
# ── 8. Optimizer & Scheduler ─────────────────────────────────────
depth_criterion = nn.MSELoss()
normal_criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG.lr, weight_decay=CFG.weight_decay, betas=(0.9, 0.999))

scheduler = WarmupThenPlateau(
    optimizer, CFG.warmup_epochs,
    warmup_lr=1e-6, base_lr=CFG.lr,
    factor=CFG.plateau_factor,
    plateau_patience=CFG.plateau_patience,
    min_lr=CFG.min_lr)

scaler = GradScaler('cuda', enabled=CFG.amp)

# Resume
start_epoch = 0
best_val_loss = float('inf')

if CFG.resume is not None:
    ckpt = torch.load(CFG.resume, map_location=device)
    model.decoder.load_state_dict(ckpt['decoder'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    scaler.load_state_dict(ckpt['scaler'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt.get('best_val_loss', float('inf'))
    print(f'Resumed from epoch {start_epoch}, best_val={best_val_loss:.4f}')

print(f'Optimizer: AdamW  lr={CFG.lr}  wd={CFG.weight_decay}')
print(f'Scheduler: {CFG.scheduler}  patience={CFG.plateau_patience}')
print(f'Regularisation: dropout={CFG.dropout}  weight_decay={CFG.weight_decay}')
print(f'Augmentation: {"TactileAugment" if CFG.tactile_augment else "None"}')

In [ ]:
# ── 9. Training Loop ──────────────────────────────────────────────
history = {'epochs': [], 'train_loss': [], 'l_depth': [],
           'l_normal': [], 'val_loss': [], 'lr': []}

log_path = osp.join(CFG.save_path, 'train_log.csv')
with open(log_path, 'w') as f:
    f.write('epoch,train_loss,l_depth,l_normal,val_loss,val_depth,val_normal,lr\n')

epochs_no_improve = 0

print('=' * 60)
print(f'DPT Training | encoder={CFG.dinov2_model}  epochs={CFG.epochs}')
print(f'  bs={CFG.batch_size}  lr={CFG.lr}  dropout={CFG.dropout}  wd={CFG.weight_decay}')
print(f'  augment={CFG.tactile_augment}  early_stop={CFG.early_stop}')
print('=' * 60)

for epoch in range(start_epoch, CFG.epochs):
    lr = scheduler.get_last_lr()[0]
    print(f'\nEpoch {epoch:03d}/{CFG.epochs-1}  lr={lr:.2e}')

    train_loss, l_depth, l_normal = train_one_epoch(
        model, train_loader, optimizer, scaler,
        depth_criterion, normal_criterion, CFG, device, epoch)

    val_loss, val_depth, val_normal = validate(
        model, val_loader, depth_criterion, normal_criterion, CFG, device)

    scheduler.step(val_loss)

    print(f'  train={train_loss:.4f}  depth={l_depth:.4f}  normal={l_normal:.4f}')
    print(f'  val={val_loss:.4f}  v_depth={val_depth:.4f}  v_normal={val_normal:.4f}')

    history['epochs'].append(epoch)
    history['train_loss'].append(train_loss)
    history['l_depth'].append(l_depth)
    history['l_normal'].append(l_normal)
    history['val_loss'].append(val_loss)
    history['lr'].append(lr)

    with open(log_path, 'a') as f:
        f.write(f'{epoch},{train_loss:.6f},{l_depth:.6f},{l_normal:.6f},'
                f'{val_loss:.6f},{val_depth:.6f},{val_normal:.6f},{lr:.2e}\n')

    # Checkpoint
    def save_ckpt(path):
        torch.save({
            'epoch': epoch,
            'decoder': model.decoder.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'scaler': scaler.state_dict(),
            'best_val_loss': best_val_loss,
            'args': vars(CFG),
        }, path)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        save_ckpt(osp.join(CFG.save_path, 'best.pth'))
        print(f'  >>> best val_loss={best_val_loss:.4f} -> best.pth')
    else:
        epochs_no_improve += 1

    if (epoch + 1) % CFG.save_every == 0:
        save_ckpt(osp.join(CFG.save_path, f'epoch_{epoch:04d}.pth'))

    save_ckpt(osp.join(CFG.save_path, 'latest.pth'))

    if len(history['epochs']) >= 2:
        plot_loss_curves(history, CFG.save_path)

    if CFG.early_stop > 0 and epochs_no_improve >= CFG.early_stop:
        print(f'\nEarly stopping: no improvement for {CFG.early_stop} epochs')
        break

print(f'\nDone. Best val loss: {best_val_loss:.4f}')
plot_loss_curves(history, CFG.save_path)

# Save decoder-only weights
dec_path = osp.join(CFG.save_path, 'dpt_decoder.pth')
torch.save(model.decoder.state_dict(), dec_path)
print(f'Decoder saved -> {dec_path}')

## Evaluation & Visualization

In [ ]:
# ── 10. Visualize Predictions ────────────────────────────────────
import matplotlib.pyplot as plt

def unnorm_normal(t):
    mu = torch.tensor(norm_mu).view(3, 1, 1)
    sd = torch.tensor(norm_std).view(3, 1, 1)
    return (t * sd + mu).clamp(0, 255).permute(1, 2, 0).numpy().astype(np.uint8)

def unnorm_depth(t):
    return (t.squeeze(0) * dmap_std[0] + dmap_mu[0]).numpy()

def unnorm_input(t):
    mu = torch.tensor(imagenet_mu).view(3, 1, 1)
    sd = torch.tensor(imagenet_std).view(3, 1, 1)
    return (t * sd + mu).clamp(0, 255).permute(1, 2, 0).numpy().astype(np.uint8)

# Load best model
best_ckpt = torch.load(osp.join(CFG.save_path, 'best.pth'), map_location=device)
model.decoder.load_state_dict(best_ckpt['decoder'])
model.eval()

NUM_VIS = 6
indices = np.linspace(0, len(val_ds) - 1, NUM_VIS, dtype=int)

fig, axes = plt.subplots(NUM_VIS, 5, figsize=(18, 3.2 * NUM_VIS))
col_titles = ['Input', 'GT Normal', 'Pred Normal', 'GT Depth', 'Pred Depth']

for row, idx in enumerate(indices):
    batch = val_ds[idx]
    img = batch['sample'].unsqueeze(0).to(device)
    cal = batch['calibration'].unsqueeze(0).to(device)

    with torch.no_grad(), autocast('cuda', enabled=CFG.amp):
        out = model(img, cal)

    pred_n = out['normal'][0].cpu()
    pred_d = out['depth'][0].cpu()
    gt_n = batch['norm']
    gt_d = batch['dmap']

    axes[row, 0].imshow(unnorm_input(batch['sample']))
    axes[row, 1].imshow(unnorm_normal(gt_n))
    axes[row, 2].imshow(unnorm_normal(pred_n))
    axes[row, 3].imshow(unnorm_depth(gt_d), cmap='viridis')
    axes[row, 4].imshow(unnorm_depth(pred_d), cmap='viridis')

    if row == 0:
        for c, t in enumerate(col_titles):
            axes[0, c].set_title(t, fontsize=11)

for ax in axes.flat:
    ax.axis('off')

plt.tight_layout()
plt.savefig(osp.join(CFG.save_path, 'val_predictions.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Visualization saved.')

In [ ]:
# ── 11. Quantitative Evaluation ──────────────────────────────────
model.eval()
normal_mse_m = AverageMeter()
depth_mse_m = AverageMeter()
angular_err_m = AverageMeter()
depth_mae_m = AverageMeter()

mu_n = torch.tensor(norm_mu).view(1, 3, 1, 1).to(device)
sd_n = torch.tensor(norm_std).view(1, 3, 1, 1).to(device)
mu_d = torch.tensor(dmap_mu).view(1, 1, 1, 1).to(device)
sd_d = torch.tensor(dmap_std).view(1, 1, 1, 1).to(device)

for batch in tqdm_nb(val_loader, desc='Evaluating'):
    imgs = batch['sample'].to(device)
    calibs = batch['calibration'].to(device)
    gt_depth = batch['dmap'].to(device)
    gt_norm = batch['norm'].to(device)
    B = imgs.size(0)

    with torch.no_grad(), autocast('cuda', enabled=CFG.amp):
        out = model(imgs, calibs)

    pred_n = out['normal'].float()
    pred_d = out['depth'].float()

    normal_mse_m.update(F.mse_loss(pred_n, gt_norm).item(), B)
    depth_mse_m.update(F.mse_loss(pred_d, gt_depth).item(), B)

    # Angular error (unnormalized normals -> unit vectors)
    pn = (pred_n * sd_n + mu_n) / 127.5 - 1.0
    gn = (gt_norm * sd_n + mu_n) / 127.5 - 1.0
    pn = F.normalize(pn, dim=1)
    gn = F.normalize(gn, dim=1)
    cos_sim = (pn * gn).sum(dim=1).clamp(-1, 1)
    angular_err_m.update(torch.acos(cos_sim).mean().item() * 180 / np.pi, B)

    # Depth MAE (unnormalized)
    pd_un = pred_d * sd_d + mu_d
    gd_un = gt_depth * sd_d + mu_d
    depth_mae_m.update(F.l1_loss(pd_un, gd_un).item(), B)

print(f'\n{"="*50}')
print(f'Evaluation Results on val objects: {val_objs}')
print(f'{"="*50}')
print(f'  Normal MSE (normalized):    {normal_mse_m.avg:.4f}')
print(f'  Depth MSE (normalized):     {depth_mse_m.avg:.4f}')
print(f'  Angular Error Mean (deg):   {angular_err_m.avg:.4f}')
print(f'  Depth MAE (unnormalized):   {depth_mae_m.avg:.4f}')

# Save metrics
metrics_path = osp.join(CFG.save_path, 'metrics.txt')
with open(metrics_path, 'w') as f:
    f.write(f'Normal MSE: {normal_mse_m.avg:.4f}\n')
    f.write(f'Depth MSE: {depth_mse_m.avg:.4f}\n')
    f.write(f'Angular Error: {angular_err_m.avg:.4f} deg\n')
    f.write(f'Depth MAE: {depth_mae_m.avg:.4f}\n')
print(f'Metrics saved -> {metrics_path}')

In [ ]:
# ── 12. Download Best Checkpoint ──────────────────────────────────
# Checkpoint is already saved to Google Drive at CFG.save_path
print(f'Checkpoints saved to Google Drive: {CFG.save_path}')
print(f'Files:')
for f in sorted(os.listdir(CFG.save_path)):
    size_mb = os.path.getsize(osp.join(CFG.save_path, f)) / 1e6
    print(f'  {f:30s} {size_mb:.1f} MB')

print(f'\nTo use on local machine:')
print(f'  1. Download best.pth from Google Drive')
print(f'  2. Run eval_sim.py:')
print(f'     python eval_sim.py --mode dinov2 --dinov2-decoder best.pth ...')
print(f'  3. Run eval_real.py:')
print(f'     python eval_real.py --dinov2-decoder best.pth ...')